# American Option Pricing with a Binomial Tree (Discrete Cash Dividends)

This notebook prices **American and European Call and Put options** on **KO (Coca-Cola)** when the company pays **fixed cash dividends on known dates** (e.g. ~$0.53 per share every quarter), using the **CRR binomial tree** with the **escrowed method** inside the tree.

**Plan:**
1. fetch KO data + risk-free rate
2. historical volatility from log returns
3. option parameters + tree functions
4. build the dividend schedule (projected from KO's payment history) and calculate $PV(D)$
5. price American & European options with and without dividends + early-exercise premium
6. implied volatility with Brent's method

## Inputs

| Input | Symbol | Where we get it |
|---|---|---|
| Current stock price | $S_0$ | Market (yfinance, latest close) |
| Strike price | $K$ | Option contract |
| Time to maturity | $T$ | Expiry date - today (in years) |
| Risk-free rate | $r$ | Treasury bill yield (`^IRX`) |
| Volatility | $\sigma$ | Historical log returns of the stock |
| **Dividend schedule** | $D_i,\ t_i$ | KO's dividend history (projected forward) |
| **Tree steps** | $N$ | Chosen by us (more steps = more accurate, slower) |

A cash dividend makes the stock price **jump down** by exactly that amount on the ex-date - the reason we need a special treatment inside the tree (explained below).

In [2]:
import numpy as np
import pandas as pd
import yfinance as yf
from datetime import datetime
import scipy.stats as stats
from scipy.optimize import brentq

# Fetch 1 year of historical daily data for KO
ticker_symbol = "KO"
df = yf.Ticker(ticker_symbol).history(period="1y")

# Extract the most recent closing price as the current stock price (S0)
S0 = df['Close'].iloc[-1]
print(f"Current {ticker_symbol} Stock Price (S0): ${S0:.2f}")

Current KO Stock Price (S0): $93.58


## Volatility from historical data

The model needs $\sigma$ = **annualized volatility** of the stock's log returns.

Daily log return:

$$r_t = \ln\!\left(\frac{P_t}{P_{t-1}}\right)$$

Log returns add up over time, so their standard deviation grows with the square root of time. With 252 trading days in a year:

$$\boxed{\sigma_{\text{annual}} = \operatorname{std}(r_t)\times\sqrt{252}}$$

**Assumption:** log returns are (roughly) normally distributed with constant volatility - this is what the binomial tree also assumes when it picks the up/down factors $u$ and $d$.

In [3]:
# Calculate daily log returns: ln(P_t / P_t-1)
df['Log_Return'] = np.log(df['Close'] / df['Close'].shift(1))

# Drop missing values if any
log_returns = df['Log_Return'].dropna()

# standard deviation and annualization
daily_volatility = log_returns.std()
historical_volatility = daily_volatility * np.sqrt(252)

print(f"Daily Volatility: {daily_volatility:.6f}")
print(f"Annualized Historical Volatility (sigma): {historical_volatility:.4f} ({historical_volatility * 100:.2f}%)")

Daily Volatility: 0.011938
Annualized Historical Volatility (sigma): 0.1895 (18.95%)


## Definition of option parameters

- **Strike** $K$ - nearest-dollar strike (at the money)
- **Time to maturity** $T = \text{days to expiry}/365.25$
- **Risk-free rate** $r$ - from `^IRX`
- **Volatility** $\sigma$ - from KO's historical log returns
- **Steps** $N = 500$

The cell also defines the two dividend helpers used later:
- `pv_remaining(t, divs, r)` - the "still to pay" pile of dividends at any time $t$
- `crr_price_discrete(...)` - the tree, built on the risky part $X_0 = S_0 - PV(D)$ (method explained after the dividend schedule)

In [4]:
# Define option parameters

K = round(S0)                # nearest-dollar strike = at the money
expiry_date = datetime(2027, 6, 18)
today = datetime.now()

# ----------------------------------------------------------------------- #

# time to maturity (T) in years
T = (expiry_date - today).days / 365.25
print(f"Strike Price (K)           : ${K}")
print(f"Time to Maturity (T)       : {T:.8f} years ({(expiry_date - today).days} days)")

# Risk-Free Rate from ^IRX and historical volatility:
irx = yf.Ticker("^IRX")
# recent close price
r = irx.history(period="5d")['Close'].iloc[-1] / 100
sigma = historical_volatility
print(f"Risk-Free Rate (r)         : {r:.4f} ({r * 100:.2f}%)")
print(f"Volatility Used (sigma)    : {sigma:.4f} ({sigma * 100:.2f}%)")

# binomial tree parameters (Cox-Ross-Rubinstein)
N = 500                      # number of steps
dt = T / N
u = np.exp(sigma * np.sqrt(dt))
d = 1.0 / u
p = (np.exp(r * dt) - d) / (u - d)
print(f"Tree Steps (N)             : {N}    dt = {dt:.6f}")
print(f"Up (u) / Down (d) / Prob (p): {u:.6f} / {d:.6f} / {p:.4f}")

def pv_remaining(t, divs, r):
    """Present value at time t of all dividends paid strictly after t."""
    return sum(D * np.exp(-r * (ti - t)) for ti, D in divs if ti > t)

def crr_price_discrete(S0, K, T, r, sigma, N, divs, option='call', style='american'):
    """
    CRR binomial tree for a stock with discrete cash dividends (escrowed method).

    The tree is built on the risky part  X0 = S0 - PV(D)  (no jumps -> recombining).
    At step time t the real stock price used for payoffs/exercise is
    S = X + PV of dividends still to be paid after t.

    option = 'call' or 'put'
    style  = 'american' or 'european'
    """
    dt = T / N
    u = np.exp(sigma * np.sqrt(dt))
    d = 1.0 / u
    p = (np.exp(r * dt) - d) / (u - d)
    disc = np.exp(-r * dt)

    PV_D = sum(D * np.exp(-r * ti) for ti, D in divs)
    X0 = S0 - PV_D                      # risky part at t = 0

    # option value at expiry (no dividends left -> stock = X)
    ST = X0 * u ** (np.arange(N, -1, -1)) * d ** (np.arange(0, N + 1))
    if option == 'call':
        vals = np.maximum(ST - K, 0.0)
    else:
        vals = np.maximum(K - ST, 0.0)

    # step backwards through the tree
    for i in range(N - 1, -1, -1):
        t = i * dt
        Xi = X0 * u ** (np.arange(i, -1, -1)) * d ** (np.arange(0, i + 1))
        Si = Xi + pv_remaining(t, divs, r)      # real stock price at time t
        vals = disc * (p * vals[:-1] + (1 - p) * vals[1:])
        if style == 'american':
            if option == 'call':
                vals = np.maximum(vals, Si - K)
            else:
                vals = np.maximum(vals, K - Si)
    return float(vals[0])

Strike Price (K)           : $94
Time to Maturity (T)       : 0.65982204 years (241 days)
Risk-Free Rate (r)         : 0.0408 (4.08%)
Volatility Used (sigma)    : 0.1895 (18.95%)
Tree Steps (N)             : 500    dt = 0.001320
Up (u) / Down (d) / Prob (p): 1.006908 / 0.993140 / 0.5022


## How to calculate the discrete dividends

**What is a discrete dividend?** A fixed amount of cash paid to shareholders on one specific ex-date - for example KO pays about $\$0.53$ per share. On the morning the stock goes ex-dividend, the price drops **down by that exact amount**. Companies usually pay **4 times a year** (quarterly).

### Where the numbers come from

1. **Past payments** - `yf.Ticker("KO").dividends` returns every past dividend (ex-date + amount).
2. **Future payments** - not available for free, so we **project**: take the latest quarterly amount and repeat it every ~3 months until the option expires. *(In practice a trader would use the company's declared schedule.)*
3. **Only dividends before expiry matter**: keep payments with $0 < t_i < T$, where

$$t_i = \frac{\text{dividend date} - \text{today}}{365.25}$$

### The two formulas the tree needs

**Present value today:**

$$PV(D) = \sum_i D_i e^{-r t_i}$$

**Present value still to come at any time $t$** (the "still to pay" pile - needed for every early-exercise decision):

$$PV_{rem}(t) = \sum_{t_j > t} D_j e^{-r (t_j - t)}, \qquad PV_{rem}(T) = 0$$

The cell below prints the schedule with each dividend's present value, plus the totals.

In [5]:
# -----------------------------------------------------------------
# Build the dividend schedule:
#   1) take past payments from Yahoo Finance
#   2) project the latest quarterly amount forward until expiry
#   3) keep only the payments that fall before the option expires
# -----------------------------------------------------------------
tk = yf.Ticker(ticker_symbol)
div_series = tk.dividends                       # past dividends: (ex-date, amount)

div_schedule = []                               # (date, cash amount) expected before expiry

if len(div_series) > 0:
    if div_series.index.tz is not None:
        div_series.index = div_series.index.tz_localize(None)

    # average number of days between payments (KO pays quarterly ~ 91 days)
    gaps = div_series.index.to_series().diff().dt.days.dropna()
    period_days = float(gaps.median()) if len(gaps) > 0 else 91.25
    last_amount = float(div_series.iloc[-1])
    last_date = div_series.index[-1]

    # project the same cash amount forward until expiry
    d = last_date + pd.Timedelta(days=period_days)
    while d < expiry_date:
        if d > today:                           # ignore payments already in the past
            div_schedule.append((d, last_amount))
        d = d + pd.Timedelta(days=period_days)

    print(f"Last actual dividend : {last_date.date()}  ${last_amount:.4f}")
    print(f"Payment period used  : {period_days:.1f} days\n")
else:
    print(f"WARNING: no dividend history found for {ticker_symbol} - using an empty schedule.")

# schedule as (time in years, cash amount)
t_years = [(dt_ - today).days / 365.25 for dt_, _ in div_schedule]
dividends = list(zip(t_years, [amt for _, amt in div_schedule]))

# present value of the dividends paid before expiry
PV_D = sum(a * np.exp(-r * t) for t, (dt_, a) in zip(t_years, div_schedule))

print("Dividend schedule before option expiry:")
print(f"{'#':>2}  {'Ex-date':<12} {'Amount':>8} {'t (years)':>10} {'PV':>8}")
for i, ((dt_, a), t) in enumerate(zip(div_schedule, t_years), start=1):
    print(f"{i:>2}  {str(dt_.date()):<12} {a:>8.4f} {t:>10.4f} {a * np.exp(-r * t):>8.4f}")
print("-" * 46)
print(f"Number of dividends : {len(dividends)}")
print(f"PV(D) = sum of D * e^(-r*t)  = ${PV_D:.4f}")
print(f"Risky part of the stock  X0 = S0 - PV(D) = ${S0 - PV_D:.4f}")

Last actual dividend : 2026-09-15  $0.5300
Payment period used  : 92.0 days

Dividend schedule before option expiry:
 #  Ex-date        Amount  t (years)       PV
 1  2026-12-16     0.5300     0.1561   0.5266
 2  2027-03-18     0.5300     0.4079   0.5213
----------------------------------------------
Number of dividends : 2
PV(D) = sum of D * e^(-r*t)  = $1.0479
Risky part of the stock  X0 = S0 - PV(D) = $92.5322


## Why we can't just subtract the dividend at a node (and what we do instead)

**The problem:** on the ex-date the stock jumps **down by $D$**. If we simply subtract $D$ from the nodes at that date, the path "up then down" no longer lands on the same price as "down then up" - after each dividend the nodes **stop recombining**, and the tree explodes (up to $2^N$ nodes).

**The standard fix - escrowed method inside the tree:**

Split the stock into two parts:

$$S(t) \;=\; \underbrace{X(t)}_{\text{risky part}} \;+\; \underbrace{PV_{rem}(t)}_{\text{cash still to be paid out}}$$

- On the ex-date **both parts drop by $D$** (the stock drops by $D$, and the "still to pay" pile shrinks by the same $D$), so $X$ **never jumps**.
- A stock that never jumps is a normal lognormal stock -> a **plain recombining CRR tree works on $X$**, starting at

$$X_0 = S_0 - PV(D)$$

- At any node with time $t$, the **real stock price** (used for the payoff and the exercise decision) is $S = X + PV_{rem}(t)$.
- At expiry the pile is empty ($PV_{rem}(T) = 0$), so the terminal payoff simply uses $X$.

**What this gives us:**
- **European** options: only the terminal payoff matters -> the tree gives the *escrowed Black-Scholes* price (validated below against the closed form).
- **American** options: at every node we compare `max(exercise with the real stock, continuation)` - exercising **just before an ex-date** becomes attractive automatically, because the holder collects the dividend (exactly what real holders do).
- This is an **approximation** (the escrowed model); exact pricing needs non-recombining trees or interpolation methods. A strike adjustment (the `'forward'` model from the European project) does **not** work here, because American options can be exercised on many different dates, not just at expiry.

## Option prices

The cell prices American and European options **with dividends** and, for comparison, **without dividends** (same tree, empty schedule):

- **dividend effect:** with dividends the call is worth *less* and the put *more* than without
- **early-exercise premium** = American $-$ European (for the put)
- **validation:** the European tree price must match the closed-form *escrowed Black-Scholes* price ($S_0$ replaced by $S_0 - PV(D)$):

$$C = (S_0 - PV(D))\,N(d_1) - Ke^{-rT}N(d_2), \qquad d_1 = \frac{\ln((S_0-PV(D))/K) + (r + \tfrac12\sigma^2)T}{\sigma\sqrt{T}}$$

In [6]:
# option prices (with the projected dividend schedule)
divs = dividends

call_amer = crr_price_discrete(S0, K, T, r, sigma, N, divs, 'call', 'american')
put_amer = crr_price_discrete(S0, K, T, r, sigma, N, divs, 'put', 'american')
call_eur = crr_price_discrete(S0, K, T, r, sigma, N, divs, 'call', 'european')
put_eur = crr_price_discrete(S0, K, T, r, sigma, N, divs, 'put', 'european')

# the same options on a stock that pays NO dividends (for comparison)
call_none = crr_price_discrete(S0, K, T, r, sigma, N, [], 'call', 'american')
put_none = crr_price_discrete(S0, K, T, r, sigma, N, [], 'put', 'american')

# validation: escrowed Black-Scholes (S0 replaced by S0 - PV(D))
S_adj = S0 - PV_D
d1 = (np.log(S_adj / K) + (r + 0.5 * sigma ** 2) * T) / (sigma * np.sqrt(T))
d2 = d1 - sigma * np.sqrt(T)
call_esc = S_adj * stats.norm.cdf(d1) - K * np.exp(-r * T) * stats.norm.cdf(d2)
put_esc = K * np.exp(-r * T) * stats.norm.cdf(-d2) - S_adj * stats.norm.cdf(-d1)

print("=" * 55)
print(f"Underlying Asset   : {ticker_symbol}")
print(f"Current Stock Price: ${S0:.2f}")
print(f"Strike Price       : ${K}")
print(f"Expiration Date    : {expiry_date}    T = {T:.4f}")
print(f"Volatility Used    : {sigma * 100:.2f}%")
print(f"Tree Steps (N)     : {N}")
print(f"Dividends          : {len(divs)}   PV(D) = ${PV_D:.4f}   X0 = ${S0 - PV_D:.4f}")
print("-" * 55)
print("WITH dividends:")
print(f"  American Call    : ${call_amer:.2f}")
print(f"  American Put     : ${put_amer:.2f}")
print(f"  European Call    : ${call_eur:.2f}")
print(f"  European Put     : ${put_eur:.2f}")
print("WITHOUT dividends (same tree):")
print(f"  American Call    : ${call_none:.2f}")
print(f"  American Put     : ${put_none:.2f}")
print("-" * 55)
print(f"Dividend effect            Call: ${call_amer - call_none:+.2f}   Put: ${put_amer - put_none:+.2f}")
print(f"Early-exercise premium     Call: ${call_amer - call_eur:.4f}   Put: ${put_amer - put_eur:.4f}")
print("-" * 55)
print(f"Validation European tree vs escrowed Black-Scholes:")
print(f"  Call: {call_eur:.4f} vs {call_esc:.4f}   diff {abs(call_eur - call_esc):.6f}")
print(f"  Put : {put_eur:.4f} vs {put_esc:.4f}   diff {abs(put_eur - put_esc):.6f}")
print("=" * 55)

Underlying Asset   : KO
Current Stock Price: $93.58
Strike Price       : $94
Expiration Date    : 2027-06-18 00:00:00    T = 0.6598
Volatility Used    : 18.95%
Tree Steps (N)     : 500
Dividends          : 2   PV(D) = $1.0479   X0 = $92.5322
-------------------------------------------------------
WITH dividends:
  American Call    : $6.17
  American Put     : $5.37
  European Call    : $6.17
  European Put     : $5.15
WITHOUT dividends (same tree):
  American Call    : $6.78
  American Put     : $4.95
-------------------------------------------------------
Dividend effect            Call: $-0.60   Put: $+0.42
Early-exercise premium     Call: $0.0000   Put: $0.2264
-------------------------------------------------------
Validation European tree vs escrowed Black-Scholes:
  Call: 6.1735 vs 6.1737   diff 0.000184
  Put : 5.1463 vs 5.1465   diff 0.000184


## Implied Volatility using Brent's root-finding method

**Normal way:** we know $\sigma$ -> compute the price (here: run the tree with the dividends). **Implied volatility:** we know the *market price* -> find the $\sigma$ that makes the tree produce it:

$$\text{find } \sigma \text{ such that } \text{Tree}_{divs}(S_0, K, T, r, \sigma) = \text{market price}$$

Same Brent root-finder as before - the pricer inside is the **binomial tree with the discrete dividend schedule**, so the dividend effect is not mistaken for volatility. We search American prices, because listed stock options are American.

In [7]:
def tree_price(CP, S0, K, sigma, T, r):
    """CRR binomial price with discrete dividends (American style), used by the root-finder."""
    if sigma <= 0:
        return 0.0
    option = 'call' if CP.upper() == 'CALL' else 'put'
    return crr_price_discrete(S0, K, T, r, sigma, N, dividends, option, 'american')

def implied_volatility(market_price, CP, S0, K, T, r):
    """
    Finds the implied volatility by reverse-engineering the binomial tree price
    using Brent's root-finding method.
    """
    # objective function: tree price - market price
    def objective_function(sigma):
        return tree_price(CP, S0, K, sigma, T, r) - market_price

    try:
        # search for volatility between 1% and 500%
        iv = brentq(objective_function, 1e-2, 5.0)
        return iv
    except ValueError:
        # returns NaN if the market price is invalid
        return np.nan

if __name__ == "__main__":
    market_price_call = 4.50    # observed market prices
    market_price_put = 3.50

    iv_call = implied_volatility(market_price_call, 'CALL', S0, K, T, r)
    iv_put = implied_volatility(market_price_put, 'PUT', S0, K, T, r)

    print()
    print(f"Market prices used : call ${market_price_call:.2f}, put ${market_price_put:.2f}")
    print()
    print(f"Implied Volatility_CALL: {iv_call:.4f} ({iv_call * 100:.2f}%)")
    print()
    print(f"Implied Volatility_PUT: {iv_put:.4f} ({iv_put * 100:.2f}%)")


Market prices used : call $4.50, put $3.50

Implied Volatility_CALL: 0.1330 (13.30%)

Implied Volatility_PUT: 0.1258 (12.58%)
